In [3]:
#preprocessing the duplicated data
import pandas as pd
import re
from rapidfuzz import fuzz
from rapidfuzz.fuzz import ratio, token_sort_ratio
from collections import defaultdict
import time

In [4]:
# === Load the raw duplicate data ===
df = pd.read_csv("smalldata_with_duplicates.csv")

# === Fill missing values with blanks ===
df.fillna('', inplace=True)

In [5]:
# === Strip whitespace and convert to lowercase ===
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip().str.lower()

# === Define automated abbreviation expansion patterns ===
abbreviation_patterns = {
    r"\bst\b": "street",
    r"\brd\b": "road",
    r"\bave\b": "avenue",
    r"\bpvt\.?\s*ltd\b": "private limited",
    r"\bltd\b": "limited",
    r"\bco\b": "company",
    r"\bcorp\b": "corporation",
    r"\binc\b": "incorporated",
    r"\btech\b": "technologies"
}

In [6]:
#Text cleaning function ===
def clean_text_field(text):
    for pattern, replacement in abbreviation_patterns.items():
        text = re.sub(pattern, replacement, text)
    text = re.sub(r'[^\w\s]', '', text)  # remove special characters
    return text

# === Apply cleaning to relevant columns ===
columns_to_standardize = [
    "Physical Address Line 1", "Organization", "Title", "Suffix",
    "Physical City", "Physical State", "Office Email", "Phone"
]

In [7]:
for col in columns_to_standardize:
    if col in df.columns:
        df[col] = df[col].astype(str).apply(clean_text_field)

# === Save preprocessed data for reference ===
df.to_csv("preprocessed_data.csv", index=False)
print(" Preprocessing done and saved as: preprocessed_data.csv")

 Preprocessing done and saved as: preprocessed_data.csv


In [8]:
# === Load preprocessed data ===
df = pd.read_csv("preprocessed_data.csv")

# === Reset index for tracking ===
df = df.reset_index()
df.rename(columns={"index": "Row"}, inplace=True)
# === Set to track duplicates ===
dedup_indices = set()
print(len(df))



1100


In [ ]:
dedup_indices = set()
start = time.time()

for i in range(1100):
    for j in range(i + 1, 1100):
        if j in dedup_indices:
            continue

        # Combine fields
        name1 = f"{df.loc[i, 'First Name']} {df.loc[i, 'Middle Name']} {df.loc[i, 'Last Name']}"
        name2 = f"{df.loc[j, 'First Name']} {df.loc[j, 'Middle Name']} {df.loc[j, 'Last Name']}"

        addr1 = f"{df.loc[i, 'Physical Address Line 1']}{df.loc[i, 'Physical City']}{df.loc[i, 'Physical State']}"
        addr2 = f"{df.loc[j, 'Physical Address Line 1']}{df.loc[j, 'Physical City']}{df.loc[j, 'Physical State']}"

        name_score = token_sort_ratio(name1, name2)
        phone_score = ratio(str(df.loc[i, 'Phone']), str(df.loc[j, 'Phone']))
        email_score = ratio(str(df.loc[i, 'Office Email']), str(df.loc[j, 'Office Email']))
        org_score = token_sort_ratio(str(df.loc[i, 'Organization']), str(df.loc[j, 'Organization']))
        addr_score = token_sort_ratio(addr1, addr2)
        total_score = (
            0.35 * name_score +
            0.32 * phone_score +
            0.2 * email_score +
            0.05 * org_score +
            0.08 * addr_score
        )

        if total_score > 80:

            dedup_indices.add(j)
            print(f"Match: Row {i} & {j} | Score: {round(total_score, 2)}")

# Drop and save
df_cleaned = df.drop(index=list(dedup_indices))
df_cleaned.to_csv("deduplicated_clean_data.csv", index=False)

print(f" Deduplication complete. Removed {len(dedup_indices)} rows")
print(" Time taken:", round(time.time() - start, 2), "seconds")


Match: Row 10 & 1057 | Score: 91.55
Match: Row 23 & 1054 | Score: 93.23
Match: Row 30 & 1055 | Score: 83.94
Match: Row 31 & 271 | Score: 85.18
Match: Row 35 & 61 | Score: 81.28
Match: Row 39 & 1077 | Score: 89.4
Match: Row 54 & 1061 | Score: 81.13
Match: Row 59 & 1039 | Score: 81.89
Match: Row 60 & 448 | Score: 81.84
Match: Row 61 & 559 | Score: 81.1
Match: Row 63 & 1098 | Score: 80.91
Match: Row 65 & 826 | Score: 80.72
Match: Row 66 & 1081 | Score: 86.9
Match: Row 67 & 1083 | Score: 91.27
Match: Row 70 & 1068 | Score: 90.85
Match: Row 76 & 1011 | Score: 90.53
Match: Row 85 & 429 | Score: 84.08
Match: Row 88 & 1097 | Score: 84.6
Match: Row 96 & 1041 | Score: 81.24
